# visualization_006

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (155).ipynb`

| Field | Value |
|---|---|
| Section | `visualization` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 10 |
| Detected functions | — |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/ipynb/trade_log_101.csv')
df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the datetime column is in datetime format
df['DateTime'] = pd.to_datetime(df['DateTime'], format='%d-%m-%Y %H:%M')

# 1. Line Plot for PnL over Time
plt.figure(figsize=(10, 6))
plt.plot(df['DateTime'], df['PnL'], label='PnL Over Time', color='blue')
plt.xlabel('DateTime')
plt.ylabel('PnL')
plt.title('PnL Over Time')
plt.legend()
plt.grid()
plt.show()


In [ ]:
# 2. Histogram for PnL Distribution
plt.figure(figsize=(8, 6))
sns.histplot(df['PnL'], kde=True, bins=50, color='green')
plt.title('PnL Distribution')
plt.xlabel('PnL')
plt.ylabel('Frequency')
plt.grid()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Calculate the rolling maximum for the balance
df['Rolling Max'] = df['Balance'].cummax()

# Create the plot
plt.figure(figsize=(10, 6))

# Plot the balance over time
plt.plot(df['DateTime'], df['Balance'], label='Balance Over Time', color='purple')

# Fill the drawdown areas with red color
plt.fill_between(
    df['DateTime'],
    df['Balance'],
    df['Rolling Max'],
    where=(df['Balance'] < df['Rolling Max']),
    color='red',
    alpha=0.3,
    label='Drawdown'
)

# Add labels and title
plt.xlabel('DateTime')
plt.ylabel('Balance')
plt.title('Balance Over Time with Drawdowns Highlighted')
plt.legend()
plt.grid()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Assuming 'df' has a 'Balance' column that tracks the portfolio balance over time
# Sort the data by DateTime to ensure proper time series ordering
df = df.sort_values(by='DateTime')

# Calculate the cumulative portfolio balance (or value) over time
df['Cumulative Balance'] = df['Balance'].cumsum()

# Calculate the rolling maximum portfolio value
df['Rolling Max'] = df['Cumulative Balance'].cummax()

# Calculate the drawdown as the difference between the cumulative balance and the rolling maximum
df['Drawdown'] = (df['Cumulative Balance'] - df['Rolling Max']) / df['Rolling Max'] * 100

# Plot the drawdown over time
plt.figure(figsize=(10, 6))
plt.plot(df['DateTime'], df['Drawdown'], label='Drawdown', color='red')
plt.xlabel('DateTime')
plt.ylabel('Drawdown (%)')
plt.title('Drawdown Over Time')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# 4. Bar Plot for Sum of PnL by Ticker
pnl_by_ticker = df.groupby('Ticker')['PnL'].sum()

plt.figure(figsize=(10, 6))
pnl_by_ticker.plot(kind='bar', color='purple')
plt.title('Sum of PnL by Ticker')
plt.xlabel('Ticker')
plt.ylabel('Sum of PnL')
plt.grid(axis='y')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# 5. Scatter Plot for PnL by Ticker with Different Colors and Adjusted Legend
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Ticker', y='PnL', data=df, hue='Ticker', palette='Set1', s=100, alpha=0.7)
plt.title('PnL for Each Ticker')
plt.xlabel('Ticker')
plt.ylabel('PnL')
plt.xticks(rotation=45)
plt.grid(True)

# Adjust the legend to avoid overlap
plt.legend(title='Ticker', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)

plt.show()


In [ ]:
# 6. Plot for Max Number of Unique Tickers Traded
import plotly.express as px

# 1. Extract the year and count unique tickers traded each day
df['Year'] = df['DateTime'].dt.year
tickers_per_day = df.groupby([df['Year'], df['DateTime'].dt.date])['Ticker'].nunique().reset_index()

# 2. Find the maximum number of unique tickers traded each day for each year
max_tickers_per_year = tickers_per_day.groupby('Year')['Ticker'].max()

# 3. Calculate the average number of tickers traded
average_tickers = max_tickers_per_year.mean()

# 4. Plot a Bar Chart with Year on the x-axis and Maximum Tickers Traded Each Day on the y-axis
fig = px.bar(max_tickers_per_year,
             labels={'value': 'Max Number of Tickers Traded', 'index': 'Year'},
             title='Max Number of Unique Tickers Traded Each Day by Year')

# 5. Add a horizontal line for the average
fig.add_shape(
    type='line',
    x0=min(max_tickers_per_year.index),
    x1=max(max_tickers_per_year.index),
    y0=average_tickers,
    y1=average_tickers,
    line=dict(color='red', dash='dash', width=2)
)

# 6. Update the layout to display the average value
fig.update_layout(
    showlegend=False,
    annotations=[dict(
        x=0.95, y=average_tickers, xref="paper", yref="y",
        text=f'Average: {average_tickers:.2f}',
        showarrow=False,
        font=dict(size=12, color='red')
    )]
)

fig.update_xaxes(title='Year')
fig.update_yaxes(title='Max Number of Unique Tickers Traded')
fig.show()


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import calendar

# Assuming 'df' has 'DateTime' and 'PnL' columns
# Ensure DateTime is in datetime format
df['DateTime'] = pd.to_datetime(df['DateTime'])

# Extract year and month
df['Year'] = df['DateTime'].dt.year
df['Month'] = df['DateTime'].dt.month

# Group by year and month to calculate normalized monthly returns (PnL sum / 1 lakh)
df['Normalized PnL'] = (df['PnL'] / 4_00_000)*100
monthly_returns = df.groupby(['Year', 'Month'])['Normalized PnL'].sum().reset_index()

# Pivot the data to create a matrix of years (rows) and months (columns)
returns_matrix = monthly_returns.pivot(index='Year', columns='Month', values='Normalized PnL')

# Replace NaN with 0 for missing months
returns_matrix = returns_matrix.fillna(0)

# Map month numbers to names for better readability
returns_matrix.columns = [calendar.month_name[m] for m in returns_matrix.columns]

# Create a larger heatmap figure
fig, ax = plt.subplots(figsize=(14, 10))

# Plot the heatmap
sns.heatmap(
    returns_matrix,
    annot=True,                      # Annotate cells with their values
    fmt=".2f",                       # Format the annotations to 2 decimal places
    linewidths=0.5,                  # Add lines between cells
    linecolor='gray',                # Line color
    cbar=False,                      # No color bar
    annot_kws={'size': 10},          # Font size for annotations
    cmap=sns.color_palette(["red", "green"], as_cmap=True),  # Red for negative, green for positive
    center=0                         # Center color on zero
)

# Add titles and labels
plt.title('Monthly Returns Heatmap (Red for Losses, Green for Gains)', fontsize=16, weight='bold')
plt.xlabel('Month', fontsize=14)
plt.ylabel('Year', fontsize=14)

# Adjust layout for better appearance
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Assuming 'df' has 'DateTime' and 'PnL' columns
# Ensure DateTime is in datetime format
df['DateTime'] = pd.to_datetime(df['DateTime'])

# Extract year
df['Year'] = df['DateTime'].dt.year

# Calculate yearly returns (sum of PnL for each year divided by 1 lakh)
df['Normalized PnL'] = (df['PnL'] / 4_00_000)*100
yearly_returns = df.groupby('Year')['Normalized PnL'].sum()

# Plot histogram for yearly returns
plt.figure(figsize=(12, 8))
plt.bar(yearly_returns.index, yearly_returns.values, color=['green' if x > 0 else 'red' for x in yearly_returns.values])
plt.xlabel('Year', fontsize=14)
plt.ylabel('Yearly Returns (in lakhs)', fontsize=14)
plt.title('Yearly Returns Histogram', fontsize=16, weight='bold')
plt.axhline(0, color='black', linewidth=1, linestyle='--')  # Add a horizontal line at y=0
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Annotate the bars with return values
for i, value in enumerate(yearly_returns.values):
    plt.text(yearly_returns.index[i], value + 0.05 if value > 0 else value - 0.05, f"{value:.2f}",
             ha='center', color='black', fontsize=10)

plt.tight_layout()
plt.show()
